In [ ]:
from pathlib import Path
import pandas as pd
from formulation.common import ProblemDataReal
from formulation.formulation_3.definition import Formulation3
from formulation.formulation_3.gurobipy import build_model_from_definition, solve_problem
from formulation.formulation_3.solution import Formulation3ModelBundle
from experiments.all_gurobi_julia import _build_run_scopes
import networkx as nx
import matplotlib.pyplot as plt
from copy import deepcopy
from itertools import product
from haversine import haversine
import numpy as np
import math

In [ ]:
cache_path = Path("formulation/cache/framingham_problem_data.pkl")
problem_data = ProblemDataReal.load_path(cache_path)
scope = _build_run_scopes(problem_data, per_school=True, per_school_type=False)

In [ ]:
avg_students_per_stop = np.average(list({stop: sum(i for i, m in enumerate(f.M) if m.stop == stop) for stop in all_stops}.values()))
total_routes = (len(all_stops) * avg_students_per_stop) / 71

math.comb(30, int(np.ceil(total_routes)))

In [ ]:
i, j = 0, 28

f = Formulation3(problem_data=scope[0].problem_data, rounds=1)
b_old = deepcopy(f.B)
m_old = deepcopy(f.M)

all_stops = sorted(
    {m.stop for m in m_old},
    key=lambda stop: (
        haversine(
            (f.S[0].geographic_location.y, f.S[0].geographic_location.x),
            (stop.geographic_location.y, stop.geographic_location.x),
        ),
        stop.node_id,
        stop.name,
    ),
)

selected_stops = set(all_stops[i:j])

f.B = b_old[:1]
f.M = [m for m in m_old if m.stop in selected_stops]

print(f"Selected stops: {len(selected_stops)}")
print(f"Total students to be served: {len(f.M)}")

p: Formulation3ModelBundle = build_model_from_definition(f)
solve_problem(p)

In [ ]:
if p.model.SolCount == 0:
    raise RuntimeError("Solve the model first; there is no incumbent solution in `p`.")

z_bq = p.variables["z_bq"]
a_mbq = p.variables["a_mbq"]
x_bqij = p.variables["x_bqij"]
T_bqi = p.variables["T_bqi"]

A_list = p.meta["A_list"]
node_to_idx = p.meta["node_to_idx"]

rows = []
for b, bus in enumerate(f.B):
    used_rounds = [q for q in range(len(f.Q)) if z_bq[b, q].X > 0.5]
    if not used_rounds:
        continue

    total_students = 0
    total_distance_km = 0.0
    arrival_summaries = []

    for q in used_rounds:
        assigned_students = [m for m in range(len(f.M)) if a_mbq[m, b, q].X > 0.5]
        total_students += len(assigned_students)
        total_distance_km += sum(
            f.d_ij(*A_list[ij])
            for ij in range(len(A_list))
            if x_bqij[b, q, ij].X > 0.5
        )

        schools_served = sorted(
            {f.M[m].school for m in assigned_students},
            key=lambda school: (school.start_time, school.name),
        )
        for school in schools_served:
            arrival = T_bqi[b, q, node_to_idx[school]].X
            arrival_summaries.append(
                f"round {q}: {school.name}, {school.start_time - arrival:.1f} min before start (T={arrival:.1f})"
            )

    rows.append(
        {
            "bus": bus.name,
            "rounds_used": len(used_rounds),
            "students": total_students,
            "distance_km": round(total_distance_km, 2),
            "arrival_vs_school_start": "; ".join(arrival_summaries),
        }
    )

report_df = pd.DataFrame(rows).sort_values("bus").reset_index(drop=True)
print(f"Buses used: {len(report_df)}")
report_df

In [ ]:
import networkx as nx

selected_stops = sorted({student.stop for student in f.M}, key=lambda stop: (stop.node_id, stop.name))
service_graph = f.problem_data.service_graph
osm_graph = f.problem_data.osm_graph
prune_m = f.problem_data.prune

stop_component_graph = nx.Graph()
stop_component_graph.add_nodes_from(stop.node_id for stop in selected_stops)
for idx, stop_a in enumerate(selected_stops):
    for stop_b in selected_stops[idx + 1:]:
        if service_graph.has_edge(stop_a.node_id, stop_b.node_id) or service_graph.has_edge(stop_b.node_id, stop_a.node_id):
            stop_component_graph.add_edge(stop_a.node_id, stop_b.node_id)

components = list(nx.connected_components(stop_component_graph))
node_to_component = {node_id: component_idx for component_idx, component in enumerate(components, start=1) for node_id in component}

component_rows = []
for component_idx, component in enumerate(components, start=1):
    component_stops = sorted([stop for stop in selected_stops if stop.node_id in component], key=lambda stop: stop.name)
    component_students = [student for student in f.M if student.stop.node_id in component]
    component_rows.append({
        "component": component_idx,
        "stops": len(component_stops),
        "students": len(component_students),
        "stop_names": "; ".join(stop.name for stop in component_stops),
    })

component_columns = ["component", "stops", "students", "stop_names"]
component_df = pd.DataFrame(component_rows, columns=component_columns).sort_values("component").reset_index(drop=True)
print(f"Selected stops: {len(selected_stops)}")
print(f"Selected students: {len(f.M)}")
print(f"Pruned service-graph stop components: {len(component_df)}")
if len(f.Q) == 1:
    print(f"With one round per bus in this solve, connectivity alone implies at least {len(component_df)} bus(es).")
display(component_df)

bridge_rows = []
for idx, stop_a in enumerate(selected_stops):
    for stop_b in selected_stops[idx + 1:]:
        if node_to_component[stop_a.node_id] == node_to_component[stop_b.node_id]:
            continue
        try:
            forward_m = nx.shortest_path_length(osm_graph, stop_a.node_id, stop_b.node_id, weight="length")
        except nx.NetworkXNoPath:
            forward_m = None
        try:
            reverse_m = nx.shortest_path_length(osm_graph, stop_b.node_id, stop_a.node_id, weight="length")
        except nx.NetworkXNoPath:
            reverse_m = None
        bridge_rows.append({
            "component_a": node_to_component[stop_a.node_id],
            "stop_a": stop_a.name,
            "component_b": node_to_component[stop_b.node_id],
            "stop_b": stop_b.name,
            "service_edge_a_to_b": service_graph.has_edge(stop_a.node_id, stop_b.node_id),
            "service_edge_b_to_a": service_graph.has_edge(stop_b.node_id, stop_a.node_id),
            "osm_a_to_b_m": None if forward_m is None else round(forward_m, 1),
            "osm_b_to_a_m": None if reverse_m is None else round(reverse_m, 1),
            "prune_m": prune_m,
            "both_directions_exceed_prune": (
                prune_m is not None
                and forward_m is not None
                and reverse_m is not None
                and forward_m > prune_m
                and reverse_m > prune_m
            ),
        })

bridge_columns = [
    "component_a",
    "stop_a",
    "component_b",
    "stop_b",
    "service_edge_a_to_b",
    "service_edge_b_to_a",
    "osm_a_to_b_m",
    "osm_b_to_a_m",
    "prune_m",
    "both_directions_exceed_prune",
]
bridge_df = pd.DataFrame(bridge_rows, columns=bridge_columns).sort_values(["component_a", "component_b", "osm_a_to_b_m", "osm_b_to_a_m"], na_position="last").reset_index(drop=True)
if bridge_df.empty:
    print("No cross-component stop pairs to inspect. The selected stops already lie in a single service-graph component.")
display(bridge_df.head(20))


In [ ]:
import osmnx as ox

if p.model.SolCount == 0:
    raise RuntimeError("Solve the model first; there is no incumbent solution in `p`.")

graph = f.problem_data.osm_graph
A_PATH = f.A_PATH
used_buses = [
    (b, bus)
    for b, bus in enumerate(f.B)
    if any(z_bq[b, q].X > 0.5 for q in range(len(f.Q)))
]

bus_route_figures = {}
for b, bus in used_buses:
    used_rounds = [q for q in range(len(f.Q)) if z_bq[b, q].X > 0.5]
    assigned_students = [
        f.M[m]
        for m in range(len(f.M))
        if any(a_mbq[m, b, q].X > 0.5 for q in used_rounds)
    ]
    route_node_ids = []

    fig, ax = ox.plot_graph(
        graph,
        show=False,
        close=False,
        node_size=0,
        edge_color="#d0d0d0",
        edge_linewidth=0.5,
        bgcolor="white",
        figsize=(10, 10),
    )

    cmap = plt.cm.get_cmap("tab10", max(1, len(used_rounds)))
    for color_idx, q in enumerate(used_rounds):
        route_color = cmap(color_idx) if len(used_rounds) > 1 else "#d94801"
        ax.plot([], [], color=route_color, linewidth=3, label=f"Round {q}")
        for ij, path in enumerate(A_list):
            if x_bqij[b, q, ij].X > 0.5:
                route_nodes = A_PATH[path]
                if len(route_nodes) >= 2:
                    route_node_ids.extend(route_nodes)
                    ox.plot_graph_route(
                        graph,
                        route_nodes,
                        ax=ax,
                        show=False,
                        close=False,
                        orig_dest_size=0,
                        route_color=route_color,
                        route_linewidth=3,
                        route_alpha=0.9,
                    )

    stop_nodes = sorted({student.stop.node_id for student in assigned_students})
    school_nodes = sorted({student.school.node_id for student in assigned_students})
    depot_node = bus.depot.node_id

    if stop_nodes:
        ax.scatter(
            [graph.nodes[node_id]["x"] for node_id in stop_nodes],
            [graph.nodes[node_id]["y"] for node_id in stop_nodes],
            c="#1f78b4",
            s=35,
            marker="o",
            label="Pickup stops",
            zorder=4,
        )
    if school_nodes:
        ax.scatter(
            [graph.nodes[node_id]["x"] for node_id in school_nodes],
            [graph.nodes[node_id]["y"] for node_id in school_nodes],
            c="#e31a1c",
            s=90,
            marker="s",
            label="Schools",
            zorder=5,
        )
    ax.scatter(
        graph.nodes[depot_node]["x"],
        graph.nodes[depot_node]["y"],
        c="black",
        s=120,
        marker="X",
        label="Depot",
        zorder=6,
    )

    if route_node_ids:
        xs = [graph.nodes[node_id]["x"] for node_id in route_node_ids]
        ys = [graph.nodes[node_id]["y"] for node_id in route_node_ids]
        x_pad = max((max(xs) - min(xs)) * 0.15, 0.005)
        y_pad = max((max(ys) - min(ys)) * 0.15, 0.005)
        ax.set_xlim(min(xs) - x_pad, max(xs) + x_pad)
        ax.set_ylim(min(ys) - y_pad, max(ys) + y_pad)

    ax.set_title(
        f"Bus {bus.name}: {len(assigned_students)} students, {len(used_rounds)} round(s)"
    )
    ax.legend(loc="best")
    bus_route_figures[bus.name] = fig
    plt.show()

In [ ]:
arrival_summaries

In [ ]:
g = f.G
visit_nodes = [f.D[0].node_id]
visit_nodes += [m.stop.node_id for m in f.M]
visit_nodes += [f.S[0].node_id]

# Check if path exists between all visit nodes
for i in range(len(visit_nodes)):
    for j in range(i + 1, len(visit_nodes)):
        if not nx.has_path(g, visit_nodes[i], visit_nodes[j]):
            print(f"No path between {visit_nodes[i]} and {visit_nodes[j]}")

In [ ]:
print(type(problem_data).__name__)
print(f"students={len(problem_data.students)}")
print(f"stops={len(problem_data.stops)}")
print(f"schools={len(problem_data.schools)}")
print(f"depots={len(problem_data.depots)}")
print(f"buses={len(problem_data.buses)}")

In [ ]:
[s for s in problem_data.students if s.attributes.special_ed or s.attributes.wheelchair_user]

In [ ]:
student_df = pd.read_csv("experiments/data/students.csv")
student_df['is_sp_ed'].any()

In [ ]:
asa_df = pd.read_excel("/Users/riccardofiorista/Documents/urops/dtemkin1/framingham-bus/data/from_framingham/All students assigned.xlsx")

In [ ]:
len(asa_df[asa_df['Student_Program']=='Default']['BUS'].unique())